In [1]:
import numpy as np
import matplotlib.pyplot as plt
from numba import njit

In [2]:
rf = np.load("/home/matmat/Documents/urban_lab/002_Superresolution/raw_data_mat/006_mouse_long/gDMAS/160_concat_IQ_rf.npy")

In [88]:
@njit
def bit_reverse(n, bits):
    result = 0
    for i in range(bits):
        if n & (1 << i):
            result |= 1 << (bits - 1 - i)
    return result

@njit(cache=True, fastmath=True)
def fft(x):
    n = len(x)
    # Check if n is a power of 2
    if n & (n-1) != 0:
        raise ValueError("Size of input must be a power of 2")
    
    # Calculate number of bits needed to represent n
    bits = 0
    temp_n = n
    while temp_n > 1:
        temp_n >>= 1
        bits += 1
    
    # Create output array
    result = np.zeros(n, dtype=np.complex128)
    
    # Bit reversal permutation
    for i in range(n):
        reversed_i = bit_reverse(i, bits)
        if i <= reversed_i:
            result[i] = x[reversed_i]
            result[reversed_i] = x[i]
        
    # Butterfly computations
    for stage in range(1, bits + 1):
        m = 1 << stage  # 2^stage
        half_m = m >> 1  # m/2
        
        for k in range(0, n, m):
            for j in range(half_m):
                # Calculate twiddle factor
                omega = np.exp(-2j * np.pi * j / m)
                
                # Apply butterfly operation
                t = omega * result[k + j + half_m]
                u = result[k + j]
                
                # Store results
                result[k + j] = u + t
                result[k + j + half_m] = u - t
    
    return result

In [92]:
%timeit -n 10 -r 3 fft(rf[0, :, 0, 0])

28.5 μs ± 2.36 μs per loop (mean ± std. dev. of 3 runs, 10 loops each)


In [100]:
fft_val

array([-2.23497627e+03+7.23727306e+02j,  5.00675696e+02+2.27814111e+02j,
       -2.85901566e+02+8.82426161e+00j,  3.93051407e+01+1.96712335e+02j,
        2.00894916e+01-6.32663916e+01j,  3.07104428e+02+2.50306371e+01j,
        2.41448091e+02+1.90546389e+01j,  1.66956160e+02+1.77637916e+02j,
        2.43259200e+02-1.19157574e+02j,  8.62227572e+01-1.39749592e+02j,
        1.44725540e+02-1.90480632e+02j, -3.79410831e+01+7.81016397e+01j,
       -1.09290539e+02+7.11987557e+01j, -1.52083902e+02-1.29838500e+01j,
       -5.07737334e+01+3.49861111e+01j,  5.74876566e+01+2.20897664e+02j,
        3.57493839e+02+1.28112322e+02j, -2.01590607e+01+1.31439585e+02j,
        9.72123454e+01+5.65740286e+01j,  1.76102477e+02+1.84039801e+02j,
        3.07685286e+02+3.38420402e+01j,  2.92717661e+02+4.33398257e+01j,
        3.38122022e+02-1.34795511e+01j,  1.44939946e+02-2.37487718e+02j,
        9.14031868e+01-2.46890710e+02j, -1.01401371e+01-3.80089283e+02j,
        4.44071844e+00+2.91441672e-01j, -2.02354326

In [99]:
%timeit -n 10 -r 3 np.fft.fft(rf[0, :, 0, 0])

29.4 μs ± 10.3 μs per loop (mean ± std. dev. of 3 runs, 10 loops each)


In [12]:
np_fft_val

array([-2.23497632e+03+7.23727295e+02j,  5.00675690e+02+2.27814117e+02j,
       -2.85901581e+02+8.82426167e+00j,  3.93051414e+01+1.96712341e+02j,
        2.00894909e+01-6.32663918e+01j,  3.07104431e+02+2.50306377e+01j,
        2.41448090e+02+1.90546398e+01j,  1.66956161e+02+1.77637909e+02j,
        2.43259201e+02-1.19157570e+02j,  8.62227554e+01-1.39749588e+02j,
        1.44725540e+02-1.90480637e+02j, -3.79410820e+01+7.81016388e+01j,
       -1.09290543e+02+7.11987534e+01j, -1.52083908e+02-1.29838505e+01j,
       -5.07737350e+01+3.49861107e+01j,  5.74876556e+01+2.20897659e+02j,
        3.57493835e+02+1.28112320e+02j, -2.01590614e+01+1.31439590e+02j,
        9.72123489e+01+5.65740280e+01j,  1.76102478e+02+1.84039795e+02j,
        3.07685272e+02+3.38420410e+01j,  2.92717651e+02+4.33398247e+01j,
        3.38122009e+02-1.34795513e+01j,  1.44939941e+02-2.37487717e+02j,
        9.14031906e+01-2.46890717e+02j, -1.01401367e+01-3.80089294e+02j,
        4.44071865e+00+2.91441679e-01j, -2.02354324